In [3]:
import pandas as pd
import re

# =========================
# FICHIERS
# =========================

corpus_file = "La-ballade-de-Pern-intégrale-sentences-lemma.csv"
characters_file = "Noms personnages de Pern.ods"

# =========================
# MOTS CIBLES
# =========================

TARGET_WORDS = [
    "amour",
    "aimer",
    "couple"
]

# =========================
# LIRE LES PERSONNAGES
# =========================

sheet = pd.read_excel(
    characters_file,
    sheet_name="Nom et genre persos",
    engine="odf"
)

characters = set()

for col in ["NOMS HUMAINS", "NOMS DRAGONS"]:
    if col in sheet.columns:
        characters.update(sheet[col].dropna().astype(str))

characters = sorted(characters)

print("Nombre de personnages :", len(characters))

# =========================
# LIRE LE CORPUS
# =========================

df = pd.read_csv(corpus_file)

if "text_lemma" not in df.columns:
    raise ValueError("La colonne text_lemma est requise.")

if "book" not in df.columns:
    raise ValueError("La colonne book est requise.")

# =========================
# PREPARER LES REGEX
# =========================

target_pattern = re.compile(
    r"\b(" + "|".join(TARGET_WORDS) + r")\b",
    re.IGNORECASE
)

character_patterns = {
    char: re.compile(r"\b" + re.escape(char) + r"\b", re.IGNORECASE)
    for char in characters
}

# =========================
# COMPTER LES CO-OCCURRENCES
# =========================

records = []

for _, row in df.iterrows():

    text = str(row["text_lemma"])
    book = row["book"]

    # combien de mots cibles
    target_count = len(target_pattern.findall(text))

    if target_count == 0:
        continue

    # vérifier les personnages
    for char, pattern in character_patterns.items():

        if pattern.search(text):

            records.append({
                "character": char,
                "book": book,
                "count": target_count
            })

# =========================
# DATAFRAME RESULTAT
# =========================

cooc_df = pd.DataFrame(records)

# =========================
# TABLEAU PERSONNAGES × TOMES
# =========================

table = cooc_df.pivot_table(
    index="character",
    columns="book",
    values="count",
    aggfunc="sum",
    fill_value=0
)

# =========================
# AJOUT COLONNE TOTAL
# =========================

# somme sur toutes les colonnes (tomes)
table["TOTAL"] = table.sum(axis=1)

# remettre TOTAL au début
cols = ["TOTAL"] + [c for c in table.columns if c != "TOTAL"]
table = table[cols]

# =========================
# EXPORT
# =========================

table.to_csv("cooccurrence_personnages_amour.csv", encoding="utf-8")

print("Tableau créé : cooccurrence_personnages_amour.csv")

Nombre de personnages : 493
Tableau créé : cooccurrence_personnages_amour.csv
